# In-context Learning: Zero-shot & Few-shot

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DSPagan/llms-time-complexity/blob/main/notebooks/in_context_learning.ipynb)

Evaluate the **base** `Llama 3.1 8B Instruct` model (no fine-tuning) on time-complexity prediction, using two zero-shot prompts, a few-shot prompt, and a chain-of-thought prompt. Metrics and confusion matrices are computed with `src/evaluate.py`.

In [ ]:
# Unsloth pulls its own compatible stack; Colab already provides a CUDA-enabled PyTorch.
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo

In [ ]:
# Clone the repo to get the code (src/) and data, then regenerate the train/test
# split deterministically from the original CodeComplex snapshot.
!git clone https://github.com/DSPagan/llms-time-complexity.git
%cd llms-time-complexity
!python src/prepare_data.py

In [ ]:
import os, sys, json, random
from collections import defaultdict

sys.path.insert(0, os.getcwd())

from unsloth import FastLanguageModel
from src.load_model import load_model
from src.evaluate import evaluate, plot_confusion_matrix, print_summary

In [ ]:
max_seq_length = 2048

model, tokenizer = load_model(max_seq_length=max_seq_length)
FastLanguageModel.for_inference(model)

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = read_jsonl("data/train_data.jsonl")
test_data = read_jsonl("data/test_data.jsonl")
print(f"train={len(train_data)}  test={len(test_data)}")

In [ ]:
# --- Zero-shot prompts ---
# Single source of truth for the label set, presented identically in every prompt.
OPTIONS = "O(1), O(logn), O(n), O(nlogn), O(n^2), O(n^3), exponential"

def prompt_direct(code):
    return (
        "Analyze the time complexity of the following code.\n"
        f"Choose exactly one of the following options: {OPTIONS}.\n"
        "Answer with only the chosen option, without any explanation.\n"
        "Code:\n"
        f"{code}"
    )

def prompt_persona(code):
    return (
        "You are an expert in algorithm analysis and time complexity.\n"
        "Analyze the time complexity of the following code.\n"
        f"Choose exactly one of the following options: {OPTIONS}.\n"
        "Answer with only the chosen option, without any explanation.\n"
        "Code:\n"
        f"{code}"
    )

ZERO_SHOT = {"direct": prompt_direct, "persona": prompt_persona}

# --- Few-shot prompt (the direct prompt + one random train example per class) ---

FEWSHOT_CLASSES = [
    ("constant", "O(1)"),
    ("logn", "O(logn)"),
    ("linear", "O(n)"),
    ("nlogn", "O(nlogn)"),
    ("quadratic", "O(n^2)"),
    ("cubic", "O(n^3)"),
    ("exponential", "exponential"),
]

_rng = random.Random(42)
_by_class = defaultdict(list)
for item in train_data:
    _by_class[item["complexity"]].append(item["src"])
fewshot_examples = {cls: _rng.choice(_by_class[cls]) for cls, _ in FEWSHOT_CLASSES}

def prompt_few_shot(code):
    parts = [
        "Analyze the time complexity of the following code.",
        f"Choose exactly one of the following options: {OPTIONS}.",
        "Answer with only the chosen option, without any explanation.",
        "Here are some examples:",
        "",
    ]
    for i, (cls, disp) in enumerate(FEWSHOT_CLASSES, start=1):
        parts += [f"Example {i}:", "Code:", fewshot_examples[cls], f"Complexity: {disp}", ""]
    parts += ["Now analyze this code:", "Code:", code, "Complexity:"]
    return "\n".join(parts)

# --- Chain-of-thought prompt (beyond the thesis: reason first, then answer) ---

def prompt_cot(code):
    return (
        "You are analyzing the worst-case time complexity of a Python program.\n\n"
        "Reason step by step:\n"
        "1. Identify the loops and recursion, and how many times each runs in terms "
        "of the input size n.\n"
        "2. Combine them to find the dominant term.\n"
        "3. Map that term to the closest complexity class.\n\n"
        f"Choose exactly one of: {OPTIONS}.\n\n"
        "After your reasoning, end with a single line exactly of the form:\n"
        "Final complexity: <chosen option>\n\n"
        "Code:\n"
        f"{code}"
    )

In [ ]:
os.makedirs("figures", exist_ok=True)

# Set LIMIT to a small number (e.g. 20) for a quick smoke test; None = full test set.
LIMIT = None
eval_data = test_data if LIMIT is None else test_data[:LIMIT]
y_true = [item["complexity"] for item in eval_data]

def generate(prompt, max_new_tokens=64):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    if inputs.shape[1] > max_seq_length:
        return None
    out = model.generate(
        input_ids=inputs, do_sample=False, max_new_tokens=max_new_tokens,
        use_cache=True, no_repeat_ngram_size=4,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("assistant")[-1].strip()

def run_prompt(build_prompt, max_new_tokens=64):
    return [generate(build_prompt(item["src"]), max_new_tokens=max_new_tokens)
            for item in eval_data]

def extract_final(text):
    # Chain-of-thought: keep only what follows the "Final complexity:" marker, so the
    # reasoning trace is not parsed instead of the conclusion.
    if text and "final complexity:" in text.lower():
        return text.lower().rsplit("final complexity:", 1)[-1]
    return text

## Zero-shot experiments

Run the three prompts over the test set and compute metrics + a confusion matrix for each.

In [ ]:
results = {}

for name, builder in ZERO_SHOT.items():
    raw = run_prompt(builder)
    res = evaluate(y_true, raw)
    results[f"zero-shot {name}"] = res
    print_summary(f"zero-shot {name}", res)
    plot_confusion_matrix(
        res["confusion_matrix"],
        title=f"Zero-shot ({name})",
        save_path=f"figures/CM_zeroshot_{name.replace(' ', '_')}.png",
    )

## Few-shot experiment

Prompt 2 (the best zero-shot prompt) plus one example per complexity class, drawn from the training set.

In [ ]:
raw = run_prompt(prompt_few_shot)
res = evaluate(y_true, raw)
results["few-shot"] = res
print_summary("few-shot", res)
plot_confusion_matrix(res["confusion_matrix"], title="Few-shot",
                      save_path="figures/CM_fewshot.png")

## Chain-of-thought (beyond the thesis)

A better-structured prompt that was **not** part of the thesis: it asks the model to reason step by step before committing to a class, and to end with a parseable `Final complexity:` line. Complexity estimation is a reasoning task, so this usually helps zero-shot the most.

In [ ]:
# CoT needs room to reason, so allow more tokens; then parse only the final line.
raw = run_prompt(prompt_cot, max_new_tokens=512)
extracted = [extract_final(r) for r in raw]
res = evaluate(y_true, extracted)
results["CoT (zero-shot)"] = res
print_summary("CoT (zero-shot)", res)
plot_confusion_matrix(res["confusion_matrix"], title="Chain-of-thought (zero-shot)",
                      save_path="figures/CM_cot.png")

## Results summary

In [ ]:
print(f"{'Experiment':<22}{'Accuracy':>10}{'Macro F1':>10}")
for name, res in results.items():
    print(f"{name:<22}{res['accuracy']:>10.3f}{res['macro_f1']:>10.3f}")